In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ==========================================
# 1. Your First nn.Module
# ==========================================

class SimpleLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)   # 1 input, 1 output

    def forward(self, x):
        return self.linear(x)

model = SimpleLinear()
print(model)
print("Parameters:", list(model.parameters()))


# TODO: Define a class called TwoLayerNet with:
#   - a first linear layer: 1 input -> 16 hidden units
#   - a ReLU activation
#   - a second linear layer: 16 hidden -> 1 output
# Then instantiate it and print the model.
class TwoLayerNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Defining individual layers explicitly
        self.fc1 = nn.Linear(1, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

two_layer_model = TwoLayerNet()
print("TwoLayerNet Structure:")
print(two_layer_model)




In [ ]:

# ==========================================
# 2. Training the MLP on a Sine Wave
# ==========================================

torch.manual_seed(0)
# Dataset: y = sin(x) with some noise
X = torch.linspace(-3.14, 3.14, 200).unsqueeze(1)
y = torch.sin(X) + 0.05 * torch.randn_like(X)

class MLP(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x)

# TODO: Change hidden=32 to hidden=4. Does the model still fit the sine wave?
#       Then try hidden=128. Write your observations as a comment.

# Test run with hidden=4
model_low_capacity = MLP(hidden=4)
# Test run with hidden=128
model_high_capacity = MLP(hidden=128)

# OBSERVATIONS:
# 1. With hidden=4: The model struggles to capture the full oscillation of the sine wave smoothly. 
#    It results under-fitted or appears "blocky/piecewise-linear" because 4 hidden units provide 
#    too few linear combinations (decision boundaries) to approximate a curved non-linear function accurately.
# 2. With hidden=128: The model fits the curve exceptionally well and converges quickly. 
#    However, if trained for too many epochs with excessively high capacity, it risks 
#    overfitting to the high-frequency random noise instead of learning just the underlying sine wave pattern.




In [ ]:

# ==========================================
# 3. Saving & Loading a Model
# ==========================================

# Baseline setup for saving
model = MLP(hidden=32)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(500):
    y_pred = model(X)
    loss = loss_fn(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Save
torch.save(model.state_dict(), "mlp_sine.pth")
print("Model saved.")

# Load
model2 = MLP(hidden=32)
model2.load_state_dict(torch.load("mlp_sine.pth"))
model2.eval()
print("Model loaded.")

# TODO: Verify the loaded model gives the same output as the original on X[:5]
# Hint: use torch.allclose()
with torch.no_grad():
    output_original = model(X[:5])
    output_loaded = model2(X[:5])

outputs_match = torch.allclose(output_original, output_loaded)
print(f"Do the original and loaded model outputs match perfectly? {outputs_match}")




In [ ]:

# ==========================================
# 4. Connecting to Scribe
# ==========================================

class SketchEncoder(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_mu  = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[0], h[1]], dim=-1)   # concatenate both directions
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

print("SketchEncoder defined — we'll implement this properly in Week 3.")